## Import Packages

In [2]:
import numpy as np
import pandas as pd

import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset

/opt/anaconda3/envs/tf-clean/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Word-Embedding Process

In [8]:
input_words = {'<SOS>': 0,
            'I': 1,
            'Love': 2,
            'Machine': 3,
            'Learning': 4,
            'And': 5,
            'Math': 6}

output_words = {'<SOS>': 0,
                '我': 1,
                '爱': 2,
                '机': 3,
                '器': 4,
                '学': 5, 
                '习': 6,
                '和': 7,
                '数': 8,
                '<EOS>':9}


inputs = torch.tensor([[1, 2, 6]])
labels = torch.tensor([[1, 2, 8, 5]])


we = nn.Embedding(num_embeddings=len(input_words),
             embedding_dim=2)
print(we(inputs))


de = nn.Embedding(num_embeddings=len(output_words),
             embedding_dim=2)
print(de(labels))

tensor([[[-0.3975, -2.2146],
         [-0.8668,  0.4354],
         [ 0.4464, -0.0126]]], grad_fn=<EmbeddingBackward0>)
tensor([[[ 1.2528,  1.0821],
         [-0.5578,  0.9210],
         [-0.2823, -0.7181],
         [-0.0991,  1.1065]]], grad_fn=<EmbeddingBackward0>)


In [9]:
we(inputs).shape

torch.Size([1, 3, 2])

In [10]:
de(labels).shape

torch.Size([1, 4, 2])

#### Encoder - Position Encoding

In [3]:
max_len = 4
d_model = 2

position = torch.arange(start=0, end = max_len, step=1).float().unsqueeze(1)
print(position)

pe = torch.zeros(max_len, d_model)
print(pe)

div_dim = 1 / torch.tensor(10000.00) ** (torch.arange(start=0, end=d_model, step=2).float() / d_model)
div_dim

pe[:, 0::2] = torch.sin(position * div_dim)
print(pe)

pe[:, 1::2] = torch.cos(position * div_dim)
print(pe)

tensor([[0.],
        [1.],
        [2.],
        [3.]])
tensor([[0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.]])
tensor([[0.0000, 0.0000],
        [0.8415, 0.0000],
        [0.9093, 0.0000],
        [0.1411, 0.0000]])
tensor([[ 0.0000,  1.0000],
        [ 0.8415,  0.5403],
        [ 0.9093, -0.4161],
        [ 0.1411, -0.9900]])


In [4]:
Encoded_Values = we(inputs) + pe
Encoded_Values

tensor([[ 0.6566, -1.0953],
        [-0.3601, -0.2567],
        [ 0.6041, -1.2913],
        [-0.3323, -0.9647]], grad_fn=<AddBackward0>)

# Encoder

#### Encoder - Self-Attention Process

In [5]:
Encoded_Values = torch.tensor([[1.16, 0.23],
                  [0.57, 1.36],
                  [4.42, -2.16]])

Query_weights = torch.tensor([[2.22, 0.41],
               [0.17, -0.51]])

Key_weights = torch.tensor([[-1.82, 0.57],
               [1.36,-0.38]])

Value_weights = torch.tensor([[-0.43, -0.59],
               [1.33, -2.15]])

In [6]:
max_len = 20

In [7]:
e_d_model = Encoded_Values.size(1)
e_d_model

2

In [8]:
Q = Encoded_Values @ Query_weights
K = Encoded_Values @ Key_weights
V = Encoded_Values @ Value_weights

In [9]:
Q

tensor([[ 2.6143,  0.3583],
        [ 1.4966, -0.4599],
        [ 9.4452,  2.9138]])

In [10]:
K

tensor([[ -1.7984,   0.5738],
        [  0.8122,  -0.1919],
        [-10.9820,   3.3402]])

In [11]:
V

tensor([[-0.1929, -1.1789],
        [ 1.5637, -3.2603],
        [-4.7734,  2.0362]])

In [12]:
Q @ K.T

tensor([[ -4.4960,   2.0546, -27.5134],
        [ -2.9554,   1.3038, -17.9718],
        [-15.3143,   7.1122, -93.9945]])

In [13]:
sim = Q @ K.T / (torch.tensor(e_d_model) ** 0.5)
sim

tensor([[ -3.1791,   1.4528, -19.4549],
        [ -2.0898,   0.9219, -12.7080],
        [-10.8289,   5.0291, -66.4642]])

In [14]:
sm = F.softmax(sim, dim =1).data.round(decimals=2)
sm

tensor([[0.0100, 0.9900, 0.0000],
        [0.0500, 0.9500, 0.0000],
        [0.0000, 1.0000, 0.0000]])

In [15]:
self_attention = sm @ V
self_attention

tensor([[ 1.5461, -3.2395],
        [ 1.4759, -3.1562],
        [ 1.5637, -3.2603]])

In [16]:
encoder = self_attention + Encoded_Values
encoder

tensor([[ 2.7061, -3.0095],
        [ 2.0459, -1.7962],
        [ 5.9837, -5.4203]])

# Decoder

### Decoder - Self-Attention

In [17]:
Decoder_Values = torch.tensor([[-2.53, 0.03],
                              [1.55, 1.27]])

D_Query_Weights = torch.tensor([[-0.19, 0.24],
                                [0.64, 1.47]])

D_Key_Weights = torch.tensor([[-0.08, 0.38],
                                [1.18, 0.67]])

D_Value_Weights = torch.tensor([[1.26, 1.10],
                                [-0.71, 0.05]])

In [18]:
D_Q = Decoder_Values @ D_Query_Weights
D_Q.round(decimals=2)

tensor([[ 0.5000, -0.5600],
        [ 0.5200,  2.2400]])

In [19]:
D_K = Decoder_Values @ D_Key_Weights
D_K.round(decimals=2)

tensor([[ 0.2400, -0.9400],
        [ 1.3700,  1.4400]])

In [20]:
D_V = Decoder_Values @ D_Value_Weights
D_V.round(decimals=2)

tensor([[-3.2100, -2.7800],
        [ 1.0500,  1.7700]])

In [21]:
D_sims = D_Q @ D_K.T / torch.sqrt(torch.tensor(e_d_model))
D_sims

tensor([[ 0.4589, -0.0874],
        [-1.4031,  2.7833]])

### Decoder - Mask

In [22]:
mask = torch.tril(torch.ones(D_sims.size(0), e_d_model))
print(mask)

mask = mask == 0
print(mask)

tensor([[1., 0.],
        [1., 1.]])
tensor([[False,  True],
        [False, False]])


In [23]:
D_sims = D_sims.masked_fill(mask=mask, value=-1e9)
D_sims

tensor([[ 4.5886e-01, -1.0000e+09],
        [-1.4031e+00,  2.7833e+00]])

In [24]:
d_attention_scores = F.softmax(D_sims, dim = 1) @ D_V
decoder = d_attention_scores + Decoder_Values
decoder

tensor([[-5.7391, -2.7515],
        [ 2.5375,  2.9704]])

# Encoder - Decoder Attention

In [25]:
encoder

tensor([[ 2.7061, -3.0095],
        [ 2.0459, -1.7962],
        [ 5.9837, -5.4203]])

In [26]:
decoder

tensor([[-5.7391, -2.7515],
        [ 2.5375,  2.9704]])

In [27]:
ED_Query_Weights = torch.tensor([[0.9, 1.32],
                                [1.00,0.38]])

ED_Key_Weights = torch.tensor([[0.94,1.28],
                               [-0.70,-0.97]])

ED_Value_Weights = torch.tensor([[-1.03,1.73],
                                 [1.11, -1.49]])

In [28]:
ED_Q = decoder @ ED_Query_Weights
ED_K = encoder @ ED_Key_Weights
ED_V = encoder @ ED_Value_Weights

In [29]:
ED_Q

tensor([[-7.9167, -8.6212],
        [ 5.2541,  4.4783]])

In [30]:
ED_K

tensor([[ 4.6504,  6.3831],
        [ 3.1805,  4.3611],
        [ 9.4189, 12.9168]])

In [31]:
ED_V

tensor([[ -6.1278,   9.1657],
        [ -4.1011,   6.2157],
        [-12.1797,  18.4281]])

In [32]:
ED_sims = ED_Q @ ED_K.T / (ED_K.size(1) ** 0.5)
ED_sims

tensor([[ -64.9444,  -44.3896, -131.4686],
        [  37.4899,   25.6259,   75.8958]])

In [33]:
ED_Attention = F.softmax(ED_sims, dim=1) @ ED_V
ED_Attention

tensor([[ -4.1011,   6.2157],
        [-12.1797,  18.4281]])

In [34]:
ED_Attention + decoder

tensor([[-9.8402,  3.4642],
        [-9.6422, 21.3984]])

2